# 01 — Data Collection (YouTube Crawl)


## 1. Dependencies

In [ ]:
!pip install -q youtube-comment-downloader pandas

import re
import time
from pathlib import Path
import pandas as pd
from youtube_comment_downloader import YoutubeCommentDownloader

## 2. Cấu hình danh sách video


In [ ]:
VIDEO_CONFIGS = [
    {"url": "https://www.youtube.com/watch?v=DpWL0sb9gpU", "topic": "giai_tri",  "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=lOzIC1kaVYE", "topic": "keo_kera",  "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=DVQWHGshx1E", "topic": "keo_kera",  "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=7evq7zQQcg8",  "topic": "keo_kera",  "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=uj_puGQf_QI",  "topic": "da_bong",   "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=i-hMUQwmAdw",  "topic": "drama",     "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=gijHNmJusG0",  "topic": "drama",     "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=D4kzSU5d4QQ",  "topic": "drama",     "limit": 4000},
    {"url": "https://www.youtube.com/watch?v=Qqyf1MCse3E",  "topic": "drama",     "limit": 4000},
    {"url": "https://www.youtube.com/watch?v=toJjgQgBwx0",  "topic": "drama",     "limit": 9000},
    {"url": "https://www.youtube.com/watch?v=EXbypBTBptM",  "topic": "drama",     "limit": 9000},
    {"url": "https://www.youtube.com/watch?v=sN2zNVHiFbQ",  "topic": "drama",     "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=4wC-CRmBRZw",  "topic": "tranh_cai", "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=jxTaydBnwIg",  "topic": "tranh_cai", "limit": 5000},
    {"url": "https://www.youtube.com/watch?v=iboJG_jom9E",  "topic": "tranh_cai", "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=KS3wBBMWMok",  "topic": "tranh_cai", "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=71fqRq6aEYw",  "topic": "tranh_cai", "limit": 1000},
    {"url": "https://www.youtube.com/watch?v=uujdjah8wAA",  "topic": "tranh_cai", "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=rjfG4Uu5UWU",  "topic": "tranh_cai", "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=dLRWw3csDKQ",  "topic": "tranh_cai", "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=ngaGpPtmJao",  "topic": "tranh_cai", "limit": 2000},
    {"url": "https://www.youtube.com/watch?v=NKIfK5VkXu4",  "topic": "tranh_cai", "limit": 5000},
]

## 3. Hàm cào bình luận


In [ ]:
def crawl_comments(url: str, limit: int) -> list[str]:
    """Cào tối đa `limit` bình luận (chỉ lấy text) từ 1 video YouTube."""
    downloader = YoutubeCommentDownloader()
    texts = []
    for i, comment in enumerate(downloader.get_comments_from_url(url)):
        if i >= limit:
            break
        texts.append(comment["text"])
    return texts

## 4. Thực thi cào theo danh sách cấu hình

In [ ]:
records = []

for cfg in VIDEO_CONFIGS:
    comments = crawl_comments(cfg["url"], cfg["limit"])
    records.extend({"text": c, "topic": cfg["topic"], "source_url": cfg["url"]} for c in comments)
    print(f"[{cfg['topic']:<10}] {cfg['url']} -> {len(comments)} comments")
    time.sleep(3)

df_raw = pd.DataFrame(records)
print(f"\nTotals: {len(df_raw):,} raw comments")

[giai_tri  ] https://www.youtube.com/watch?v=DpWL0sb9gpU -> 2000 comments
[keo_kera  ] https://www.youtube.com/watch?v=lOzIC1kaVYE -> 542 comments
[keo_kera  ] https://www.youtube.com/watch?v=DVQWHGshx1E -> 357 comments
[keo_kera  ] https://www.youtube.com/watch?v=7evq7zQQcg8 -> 169 comments
[da_bong   ] https://www.youtube.com/watch?v=uj_puGQf_QI -> 1733 comments
[drama     ] https://www.youtube.com/watch?v=i-hMUQwmAdw -> 1199 comments
[drama     ] https://www.youtube.com/watch?v=gijHNmJusG0 -> 1213 comments
[drama     ] https://www.youtube.com/watch?v=D4kzSU5d4QQ -> 4000 comments
[drama     ] https://www.youtube.com/watch?v=Qqyf1MCse3E -> 3992 comments
[drama     ] https://www.youtube.com/watch?v=toJjgQgBwx0 -> 8407 comments
[drama     ] https://www.youtube.com/watch?v=EXbypBTBptM -> 3636 comments
[drama     ] https://www.youtube.com/watch?v=sN2zNVHiFbQ -> 0 comments
[tranh_cai ] https://www.youtube.com/watch?v=4wC-CRmBRZw -> 528 comments
[tranh_cai ] https://www.youtube.com/watch?v=

## 5. Làm sạch dữ liệu (Cleaning Pipeline)


In [ ]:
VIETNAMESE_CHARS = set("àáâãèéêìíòóôõùúýăđơưạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹ")

def remove_urls(text: str) -> str:
    return re.sub(r"http\S+|www\S+", "", text).strip()

def is_vietnamese(text: str, min_chars: int = 2) -> bool:
    return sum(c in VIETNAMESE_CHARS for c in text.lower()) >= min_chars

def clean_comments(df: pd.DataFrame, min_words: int = 3) -> pd.DataFrame:
    df = df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).copy()
    df["text"] = df["text"].apply(remove_urls)
    df = df[df["text"].apply(is_vietnamese)]
    df = df[df["text"].str.split().str.len() >= min_words]
    return df.reset_index(drop=True)

df_clean = clean_comments(df_raw)
print(f"Before cleaning: {len(df_raw):,} -> After: {len(df_clean):,}")

Before cleaning: 43,171 -> After: 38,128


## 6. Lưu kết quả

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

df_clean.to_csv(RAW_DIR / "tdtu_youtube.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df_clean):,} comments to {RAW_DIR / 'tdtu_youtube.csv'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved 38,128 comments to /content/drive/MyDrive/Hate_Speech_Detection/data/raw/tdtu_youtube.csv


## Tổng kết

In [ ]:
df_clean["topic"].value_counts()

,count
topic,
drama,19800
tranh_cai,14463
giai_tri,1824
da_bong,1093
keo_kera,948
